## **Step 3: XAI — Explainability & Traceability**

**Project:** *Towards Explainable and Reliable Large Language Models for Clinical Decision Support*

### Notebook objective

Provide a **post-hoc explainability and traceability layer** over the outputs of Step 1 (RAG retrieval & generation) and Step 2 (hallucination control).  This module decomposes each generated answer into an auditable three-level XAI structure — question-level dashboard, claim-level explanations, and evidence-level attribution — and produces deterministic, reproducible visual summaries without any additional LLM calls.

### XAI levels

| Level | Dataframe | Grain | Key fields |
| --- | --- | --- | --- |
| 1 — Question dashboard | `xai_answer_dashboard_df` | 1 row / question | risk_level, final_answer_status, recommended_action |
| 2 — Claim explanations | `xai_claim_explanations_df` | 1 row / claim | verification_label, final_status, mitigation_path |
| 3 — Evidence attribution | `xai_evidence_attribution_df` | 1 row / evidence | nli_score, bio_similarity, evidence_text |

### Input data inventory

| Source file | Rows | Key columns consumed |
| --- | --- | --- |
| `answer_summary.csv` | 100 | question_id, total_claims, supported/contradicted/neutral_claims, support_rate, hallucination_risk |
| `mitigated_rag_answers.csv` | 100 | question_id, original_answer, mitigated_answer, gold_long_answer, resolved/corrected/reretrieved/human_review_claims, mitigation_success_rate |
| `claim_verification.csv` | 644 | question_id, claim_id, claim, verification_label, nli_score, bio_similarity, action, reason |
| `mitigation_trace.csv` | 644 | question_id, claim_id, original_claim, final_claim, final_action, final_status, attempts, final_evidence_chunk_id |
| `ablation_comparison.csv` | 10 | metric, without_control, with_control, delta |
| `human_expert_review_log.csv` | 69 | question_id, claim_id, claim, final_status, reviewer_notes |
| `faithfulness_sentence_level_results.csv` | 2936 | question, sentence, label, score, p_entailment, p_contradiction |
| `final_selected_configuration_metrics.csv` | 1 | faithfulness, rouge1_f1_cleaned, bleu_cleaned, bertscore_f1_cleaned |



---
# 5.0 Imports and Configuration

In [ ]:
# Section 5.0 - Imports and Configuration
import json
import os
import re
import textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
BASE_DIR          = Path.cwd()
OUTPUTS_DIR       = BASE_DIR / "data" / "outputs"
PIPELINE_DIR      = OUTPUTS_DIR / "final_rag_pipeline_v5"
HALL_CONTROL_DIR  = OUTPUTS_DIR / "hallucination_control"
XAI_DIR           = OUTPUTS_DIR / "xai"
XAI_DIR.mkdir(parents=True, exist_ok=True)

# Input files
ANSWER_SUMMARY_CSV       = HALL_CONTROL_DIR / "answer_summary.csv"
MITIGATED_ANSWERS_CSV    = HALL_CONTROL_DIR / "mitigated_rag_answers.csv"
CLAIM_VERIFICATION_CSV   = HALL_CONTROL_DIR / "claim_verification.csv"
MITIGATION_TRACE_CSV     = HALL_CONTROL_DIR / "mitigation_trace.csv"
ABLATION_CSV             = HALL_CONTROL_DIR / "ablation_comparison.csv"
HUMAN_REVIEW_CSV         = HALL_CONTROL_DIR / "human_expert_review_log.csv"
FAITHFULNESS_CSV         = PIPELINE_DIR / "faithfulness_sentence_level_results.csv"
CONFIG_METRICS_CSV       = PIPELINE_DIR / "final_selected_configuration_metrics.csv"

# Output files
XAI_DASHBOARD_CSV        = XAI_DIR / "xai_answer_dashboard.csv"
XAI_CLAIMS_CSV           = XAI_DIR / "xai_claim_explanations.csv"
XAI_EVIDENCE_CSV         = XAI_DIR / "xai_evidence_attribution.csv"
XAI_REPORT_TXT           = XAI_DIR / "xai_report.txt"

# ------------------------------------------------------------------
# Risk thresholds (aligned with Notebook 2 hallucination_control module)
# Notebook 2 escalation: UNRESOLVED_HUMAN_REVIEW claims go to human experts.
# XAI maps those claim-level outcomes to question-level risk:
#   HIGH  = needs_human_review OR hallucination_risk >= 0.5
#   MEDIUM = hallucination_risk >= 0.2  (claims needed mitigation but resolved)
#   LOW   = hallucination_risk < 0.2    (all claims directly supported)
RISK_HIGH_THRESHOLD   = 0.5
RISK_MEDIUM_THRESHOLD = 0.2

# NOTE: These thresholds are descriptive categories used for XAI visualization
# and do NOT represent calibrated clinical probabilities.

print("XAI module paths configured.")
print(f"  Pipeline outputs : {PIPELINE_DIR}")
print(f"  Hall. control    : {HALL_CONTROL_DIR}")
print(f"  XAI outputs      : {XAI_DIR}")


XAI module paths configured.
  Pipeline outputs : d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\final_rag_pipeline_v5
  Hall. control    : d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control
  XAI outputs      : d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\xai


---
# 5.1 Load Upstream Data

In [ ]:
# Section 5.1 - Load all upstream data (warn-and-continue for missing files)
def _safe_read_csv(path, label):
    """Read a CSV; if missing, warn and return None."""
    if not path.exists():
        warnings.warn(f"{label}: {path.name} not found — skipping.", stacklevel=2)
        return None
    return pd.read_csv(path)

answer_summary_df  = _safe_read_csv(ANSWER_SUMMARY_CSV,  "answer_summary")
mitigated_df       = _safe_read_csv(MITIGATED_ANSWERS_CSV, "mitigated_answers")
claim_verify_df    = _safe_read_csv(CLAIM_VERIFICATION_CSV, "claim_verification")
mitigation_df      = _safe_read_csv(MITIGATION_TRACE_CSV, "mitigation_trace")
ablation_df        = _safe_read_csv(ABLATION_CSV, "ablation_comparison")
human_review_df    = _safe_read_csv(HUMAN_REVIEW_CSV, "human_review_log")
faithfulness_df    = _safe_read_csv(FAITHFULNESS_CSV, "faithfulness")
config_metrics_df  = _safe_read_csv(CONFIG_METRICS_CSV, "config_metrics")

# Gate: claim_verify and mitigation_trace are mandatory for Levels 2-3
if claim_verify_df is None or mitigation_df is None:
    raise RuntimeError(
        "claim_verification.csv and mitigation_trace.csv are required. "
        "Run Notebook 2 (hallucination control) first."
    )

_loaded = {"answer_summary": answer_summary_df, "mitigated": mitigated_df,
           "claim_verify": claim_verify_df, "mitigation_trace": mitigation_df,
           "ablation": ablation_df, "human_review": human_review_df,
           "faithfulness": faithfulness_df, "config_metrics": config_metrics_df}
for name, df in _loaded.items():
    status = f"{df.shape}" if df is not None else "SKIPPED"
    print(f"  {name:25s}: {status}")


  answer_summary           : (100, 11)
  mitigated                : (100, 15)
  claim_verify             : (644, 9)
  mitigation_trace         : (644, 11)
  ablation                 : (10, 4)
  human_review             : (69, 10)
  faithfulness             : (2936, 9)
  config_metrics           : (1, 30)


---
# 5.2 Level 1 — Question-Level XAI Dashboard

Merge `answer_summary`, `mitigated_rag_answers`, and sentence-level `faithfulness` into a single
per-question dashboard dataframe (`xai_answer_dashboard_df`) and derive deterministic fields:
`risk_level`, `final_answer_status`, `recommended_action`, and natural-language summaries.


In [ ]:
# Section 5.2 - Build xai_answer_dashboard_df (Level 1)
# ---- merge answer_summary + mitigated_rag_answers on question_id ----
xai_answer_dashboard_df = answer_summary_df.merge(
    mitigated_df[["question_id", "original_answer", "mitigated_answer",
                   "gold_long_answer", "gold_final_decision", "gold_pmid",
                   "resolved_claims", "corrected_claims", "reretrieved_claims",
                   "human_review_claims", "needs_human_review",
                   "mitigation_success_rate", "unresolved_rate"]],
    on="question_id", how="left",
)

# ---- sentence-level faithfulness aggregated per question ----
faith_agg = faithfulness_df.groupby("question").agg(
    n_supported=("label", lambda s: (s == "supported").sum()),
    n_partially=("label", lambda s: (s == "partially_supported").sum()),
    n_unsupported=("label", lambda s: (s == "unsupported").sum()),
    n_total=("label", "count"),
    avg_p_entail=("p_entailment", "mean"),
    avg_p_contra=("p_contradiction", "mean"),
).reset_index()
faith_agg["sent_faithfulness"] = faith_agg["n_supported"] / faith_agg["n_total"]

xai_answer_dashboard_df = xai_answer_dashboard_df.merge(
    faith_agg[["question", "sent_faithfulness", "n_supported", "n_partially", "n_unsupported", "n_total",
               "avg_p_entail", "avg_p_contra"]],
    left_on="question", right_on="question", how="left",
)

# ---- derived fields (deterministic) ----
def _risk_level(row):
    if row["needs_human_review"]:
        return "HIGH"
    if row["hallucination_risk"] >= RISK_HIGH_THRESHOLD:
        return "HIGH"
    if row["hallucination_risk"] >= RISK_MEDIUM_THRESHOLD:
        return "MEDIUM"
    return "LOW"

def _answer_status(row):
    if row["needs_human_review"]:
        return "REVIEW_NEEDED"
    if row["unresolved_rate"] > 0:
        return "PARTIALLY_RESOLVED"
    if row["corrected_claims"] > 0 or row["reretrieved_claims"] > 0:
        return "MITIGATED"
    return "CLEAN"

def _recommended_action(row):
    rl = row["risk_level"]
    st = row["final_answer_status"]
    if rl == "HIGH" and st == "REVIEW_NEEDED":
        return "HUMAN_REVIEW_REQUIRED"
    if rl == "HIGH":
        return "FLAG_AND_REVIEW"
    if rl == "MEDIUM":
        return "MONITOR"
    return "ACCEPT"

def _explanation_summary(row):
    parts = []
    parts.append(f"{row['total_claims']} claims identified; {row['supported_claims']} supported, "
                 f"{row['contradicted_claims']} contradicted, {row['neutral_claims']} unresolved.")
    if row.get("corrected_claims", 0) > 0:
        parts.append(f"{row['corrected_claims']} claim(s) corrected via evidence-grounded rewrite.")
    if row.get("reretrieved_claims", 0) > 0:
        parts.append(f"{row['reretrieved_claims']} claim(s) resolved through re-retrieval.")
    if row.get("human_review_claims", 0) > 0:
        parts.append(f"{row['human_review_claims']} claim(s) escalated to human review.")
    return " ".join(parts)

def _plain_language(row):
    rl = row["risk_level"]
    sc = row["supported_claims"]
    tc = row["total_claims"]
    hr = row["needs_human_review"]
    if rl == "LOW":
        return (f"All {tc} claims are supported by evidence. "
                "No unresolved claim was detected in the available evidence trace.")
    if rl == "MEDIUM":
        return (f"{sc} of {tc} claims are directly supported. "
                f"{tc - sc} claim(s) required correction or re-retrieval. "
                "Review recommended before relying on the answer.")
    # HIGH
    if hr:
        return (f"{sc} of {tc} claims are supported; {row['human_review_claims']} claim(s) "
                "could not be automatically resolved and require expert review.")
    return (f"{sc} of {tc} claims are supported. "
            "Multiple claims required intervention. Careful review essential.")

xai_answer_dashboard_df["risk_level"]          = xai_answer_dashboard_df.apply(_risk_level, axis=1)
xai_answer_dashboard_df["final_answer_status"] = xai_answer_dashboard_df.apply(_answer_status, axis=1)
xai_answer_dashboard_df["recommended_action"]  = xai_answer_dashboard_df.apply(_recommended_action, axis=1)
xai_answer_dashboard_df["explanation_summary"] = xai_answer_dashboard_df.apply(_explanation_summary, axis=1)
xai_answer_dashboard_df["plain_language"]      = xai_answer_dashboard_df.apply(_plain_language, axis=1)

print(f"xai_answer_dashboard_df shape: {xai_answer_dashboard_df.shape}")
print(f"Columns: {list(xai_answer_dashboard_df.columns)}")
print()
print(xai_answer_dashboard_df[["question_id", "total_claims", "hallucination_risk",
            "risk_level", "final_answer_status", "recommended_action"]].head(10).to_string())


xai_answer_dashboard_df shape: (100, 35)
Columns: ['question_id', 'question', 'generated_answer', 'total_claims', 'supported_claims', 'contradicted_claims', 'neutral_claims', 'support_rate', 'contradiction_rate', 'missing_evidence_rate', 'hallucination_risk', 'original_answer', 'mitigated_answer', 'gold_long_answer', 'gold_final_decision', 'gold_pmid', 'resolved_claims', 'corrected_claims', 'reretrieved_claims', 'human_review_claims', 'needs_human_review', 'mitigation_success_rate', 'unresolved_rate', 'sent_faithfulness', 'n_supported', 'n_partially', 'n_unsupported', 'n_total', 'avg_p_entail', 'avg_p_contra', 'risk_level', 'final_answer_status', 'recommended_action', 'explanation_summary', 'plain_language']

   question_id  total_claims  hallucination_risk risk_level final_answer_status     recommended_action
0            0             2            0.500000       HIGH           MITIGATED        FLAG_AND_REVIEW
1            1             8            0.000000        LOW               C

---
# 5.3 Level 2 — Claim-Level Explanations

Join `claim_verification` and `mitigation_trace` to produce `xai_claim_explanations_df`,
one row per claim, with the mitigation path, number of attempts, and final resolution status.


In [ ]:
# Section 5.3 - Build xai_claim_explanations_df (Level 2)
# ---- join claim_verification + mitigation_trace ----
claims = claim_verify_df.merge(
    mitigation_df[["claim_id", "final_claim", "final_action", "final_status",
                    "final_evidence_chunk_id", "final_evidence_text", "attempts"]],
    on="claim_id", how="left",
    suffixes=("", "_trace"),
)

# ---- parse attempts JSON ----
def _parse_attempts(raw):
    if pd.isna(raw) or raw == "[]":
        return []
    try:
        parsed = json.loads(raw) if isinstance(raw, str) else raw
        return parsed if isinstance(parsed, list) else []
    except (json.JSONDecodeError, TypeError):
        return []

claims["attempts_list"]      = claims["attempts"].apply(_parse_attempts)
claims["num_attempts"]       = claims["attempts_list"].apply(len)
claims["retrieval_queries"]  = claims["attempts_list"].apply(
    lambda a: [at.get("query", at.get("queries", [""])[0] if isinstance(at.get("queries"), list) and at.get("queries") else "") for at in a]
)

# ---- evidence source type ----
def _evidence_source(row):
    if row["final_status"] in ("RESOLVED_SUPPORTED",):
        return "original_evidence"
    if row["final_status"] in ("RESOLVED_GROUNDED",):
        return "grounding_rewrite"
    if row["final_status"] in ("RESOLVED_AFTER_RERETRIEVAL", "RESOLVED_CORRECTED_AFTER_RERETRIEVAL"):
        return "re_retrieved_evidence"
    if row["final_status"] in ("RESOLVED_CORRECTED",):
        return "corrected_claim"
    if row["final_status"] == "UNRESOLVED_HUMAN_REVIEW":
        return "no_sufficient_evidence"
    return "unknown"

claims["evidence_source_type"] = claims.apply(_evidence_source, axis=1)

# ---- mitigation path summary ----
def _mitigation_path(row):
    if row["num_attempts"] == 0:
        return "no_mitigation_needed"
    actions = [a.get("action", a.get("final_action", "")) for a in row["attempts_list"]]
    if "re_retrieve" in actions:
        return "re_retrieval"
    if "correct" in actions:
        return "correction"
    return "other"

claims["mitigation_path"] = claims.apply(_mitigation_path, axis=1)

# ---- clean up for output ----
xai_claim_explanations_df = claims[[
    "question_id", "claim_id", "claim", "verification_label", "nli_score", "bio_similarity",
    "best_evidence_chunk_id", "action", "reason",
    "final_claim", "final_action", "final_status", "final_evidence_chunk_id",
    "evidence_source_type", "num_attempts", "mitigation_path", "retrieval_queries",
]].copy()

print(f"xai_claim_explanations_df shape: {xai_claims.shape}")
print(f"Columns: {list(xai_claim_explanations_df.columns)}")
print()
print("final_status distribution:")
print(xai_claim_explanations_df["final_status"].value_counts().to_string())
print()
print("evidence_source_type distribution:")
print(xai_claim_explanations_df["evidence_source_type"].value_counts().to_string())


NameError: name 'xai_claims' is not defined

---
# 5.4 Level 3 — Evidence Attribution

Connect each claim to its supporting evidence chunk, including similarity scores,
a truncated evidence excerpt, and the **full evidence text** (`evidence_full_text`)
preserved for traceability.  This produces `xai_evidence_attribution_df` for
evidence-level auditing.


In [ ]:
# Section 5.4 - Build xai_evidence_attribution_df (Level 3)
ev = claims[[
    "question_id", "claim_id", "claim", "verification_label",
    "best_evidence_chunk_id", "nli_score", "bio_similarity", "reason",
    "final_evidence_chunk_id", "final_evidence_text", "final_status",
    "evidence_source_type",
]].copy()

# full evidence text preserved for traceability
ev["evidence_full_text"] = ev["final_evidence_text"]

# truncated excerpt for display purposes only
MAX_EVID_LEN = 300
ev["evidence_excerpt"] = ev["final_evidence_text"].apply(
    lambda t: (str(t)[:MAX_EVID_LEN] + "...") if isinstance(t, str) and len(str(t)) > MAX_EVID_LEN else t
)

# evidence alignment score: average of nli_score and bio_similarity
ev["evidence_alignment_score"] = ev[["nli_score", "bio_similarity"]].mean(axis=1)

xai_evidence_attribution_df = ev.copy()

print(f"xai_evidence_attribution_df shape: {xai_evidence_attribution_df.shape}")
print(f"Columns: {list(xai_evidence_attribution_df.columns)}")
print()
print("evidence_alignment_score statistics:")
print(xai_evidence_attribution_df["evidence_alignment_score"].describe().to_string())


xai_evidence_attribution_df shape: (644, 15)
Columns: ['question_id', 'claim_id', 'claim', 'verification_label', 'best_evidence_chunk_id', 'nli_score', 'bio_similarity', 'reason', 'final_evidence_chunk_id', 'final_evidence_text', 'final_status', 'evidence_source_type', 'evidence_full_text', 'evidence_excerpt', 'evidence_alignment_score']

evidence_alignment_score statistics:
count    644.000000
mean       0.920487
std        0.065060
min        0.686120
25%        0.906185
50%        0.943490
75%        0.963787
max        0.990195


---
# 5.5 Risk Stratification Analysis

Stratify the 100 questions by `risk_level` and compute aggregate statistics per stratum.

**Risk thresholds** (defined in Section 5.0, aligned with Notebook 2 hallucination control):
- `HIGH`: `hallucination_risk >= 0.5` **or** any claims escalated to human review
- `MEDIUM`: `hallucination_risk >= 0.2` (claims needed mitigation but all resolved)
- `LOW`: `hallucination_risk < 0.2` (all claims directly supported)

> **Disclaimer:** These thresholds are descriptive categories used for XAI visualization and do **not** represent calibrated clinical probabilities.



In [ ]:
# Section 5.5 - Risk Stratification Analysis
risk_order = ["LOW", "MEDIUM", "HIGH"]
risk_palette = {"LOW": "#2ecc71", "MEDIUM": "#f39c12", "HIGH": "#e74c3c"}

risk_stats = xai_answer_dashboard_df.groupby("risk_level", observed=False).agg(
    n_questions=("question_id", "count"),
    mean_claims=("total_claims", "mean"),
    mean_support_rate=("support_rate", "mean"),
    mean_contradiction_rate=("contradiction_rate", "mean"),
    mean_hallucination_risk=("hallucination_risk", "mean"),
    mean_mitigation_success=("mitigation_success_rate", "mean"),
    n_human_review=("needs_human_review", "sum"),
).reindex(risk_order)

risk_stats["share"] = (risk_stats["n_questions"] / risk_stats["n_questions"].sum() * 100).round(1)

print("Risk stratification summary:")
print(risk_stats.to_string())
print()

# also compute answer-level HR by risk level (from ablation-style counting)
for level in risk_order:
    sub = xai_answer_dashboard_df[xai_answer_dashboard_df["risk_level"] == level]
    total = len(sub)
    if total == 0:
        continue
    flagged = sub["needs_human_review"].sum()
    print(f"  {level}: {total} questions, {flagged} flagged for human review "
          f"({flagged/total*100:.1f}%)")


Risk stratification summary:
            n_questions  mean_claims  mean_support_rate  mean_contradiction_rate  mean_hallucination_risk  mean_mitigation_success  n_human_review  share
risk_level                                                                                                                                               
LOW                  49     6.326531           0.972811                 0.005669                 0.027189                 1.000000               0   49.0
MEDIUM               12     4.583333           0.690972                 0.010417                 0.309028                 1.000000               0   12.0
HIGH                 39     7.153846           0.407149                 0.069319                 0.592851                 0.743996              32   39.0

  LOW: 49 questions, 0 flagged for human review (0.0%)
  MEDIUM: 12 questions, 0 flagged for human review (0.0%)
  HIGH: 39 questions, 32 flagged for human review (82.1%)


---
# 5.6 Single-Answer XAI Case Study

Select one representative question (a MEDIUM-risk case with both correction and re-retrieval)
and display its full three-level XAI trace.


In [ ]:
# Section 5.6 - Single-Answer XAI Case Study
# ---- select a representative case ----
# Prefer: re-retrieved + corrected (richest mitigation trace)
candidates = xai_answer_dashboard_df[
    (xai_answer_dashboard_df["reretrieved_claims"] > 0) &
    (xai_answer_dashboard_df["corrected_claims"] > 0)
].copy()

if len(candidates) == 0:
    # fallback: any question with re-retrieval
    candidates = xai_answer_dashboard_df[xai_answer_dashboard_df["reretrieved_claims"] > 0].copy()
if len(candidates) == 0:
    # fallback: any question with corrections
    candidates = xai_answer_dashboard_df[xai_answer_dashboard_df["corrected_claims"] > 0].copy()
if len(candidates) == 0:
    # fallback: highest hallucination risk
    candidates = xai_answer_dashboard_df.nlargest(5, "hallucination_risk")

case_qid = int(candidates.iloc[0]["question_id"])
print(f"Selected case study: question_id = {case_qid}")
print("=" * 80)

# ---- Level 1: Question dashboard ----
row = xai_answer_dashboard_df[xai_answer_dashboard_df["question_id"] == case_qid].iloc[0]
print(f"\n[LEVEL 1 - QUESTION DASHBOARD]")
print(f"  Question          : {row['question'][:200]}")
print(f"  Gold answer       : {str(row.get('gold_long_answer', ''))[:200]}")
print(f"  Mitigated answer  : {str(row.get('mitigated_answer', ''))[:200]}")
print(f"  Total claims      : {row['total_claims']}")
print(f"  Supported         : {row['supported_claims']}")
print(f"  Contradicted      : {row['contradicted_claims']}")
print(f"  Neutral/missing   : {row['neutral_claims']}")
print(f"  Hallucination risk: {row['hallucination_risk']:.3f}")
print(f"  Risk level        : {row['risk_level']}")
print(f"  Answer status     : {row['final_answer_status']}")
print(f"  Recommended       : {row['recommended_action']}")
print(f"  Explanation       : {row['explanation_summary']}")
print(f"  Plain language    : {row['plain_language']}")

# ---- Level 2: Claim explanations ----
case_claims = xai_claim_explanations_df[xai_claim_explanations_df["question_id"] == case_qid].copy()
print(f"\n[LEVEL 2 - CLAIM EXPLANATIONS] ({len(case_claims)} claims)")
for _, cr in case_claims.iterrows():
    print(f"  [{cr['claim_id']}] {cr['claim'][:120]}...")
    print(f"    Initial label : {cr['verification_label']}  (NLI={cr['nli_score']:.3f}, BioSim={cr['bio_similarity']:.3f})")
    print(f"    Final status  : {cr['final_status']}")
    print(f"    Evidence src  : {cr['evidence_source_type']}  (attempts={cr['num_attempts']})")
    if cr["num_attempts"] > 0:
        print(f"    Mitigation    : {cr['mitigation_path']}")
    print()

# ---- Level 3: Evidence attribution ----
case_ev = xai_evidence_attribution_df[xai_evidence_attribution_df["question_id"] == case_qid].copy()
print(f"[LEVEL 3 - EVIDENCE ATTRIBUTION] ({len(case_ev)} records)")
for _, er in case_ev.iterrows():
    excerpt = str(er.get("evidence_excerpt", ""))[:150]
    print(f"  [{er['claim_id']}] chunk={er['final_evidence_chunk_id']}  "
          f"alignment={er['evidence_alignment_score']:.3f}  status={er['final_status']}")
    if excerpt and excerpt != "nan":
        print(f"    Evidence: {excerpt}...")
    print()


Selected case study: question_id = 0

[LEVEL 1 - QUESTION DASHBOARD]
  Question          : Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?
  Gold answer       : Pap smears with glandular cell abnormalities are often flagged as abnormal by RPS, and this results in a sensitivity of 36.4% (at the AGC threshold). Most importantly, some cases of AGC are detected o
  Mitigated answer  : 36.4% of Pap smears with glandular cell abnormalities were flagged as "review for abnormality" on rapid prescreening. Thirteen of 24 Pap smears (54.2%) from patients who had histologic follow-up and w
  Total claims      : 2
  Supported         : 1
  Contradicted      : 0
  Neutral/missing   : 1
  Hallucination risk: 0.500
  Risk level        : HIGH
  Answer status     : MITIGATED
  Recommended       : FLAG_AND_REVIEW
  Explanation       : 2 claims identified; 1 supported, 0 contradicted, 1 unresolved. 1 claim(s) corrected via evidence-grounded rewrite. 1 claim(s) resolv

---
# 5.7 NL Explanation Summaries

Generate deterministic, template-based natural-language explanations for every question.
No LLM calls — purely rule-based text from the claim and mitigation statistics.


In [ ]:
# Section 5.7 - Deterministic NL Explanation Summaries

def generate_nl_explanation(row):
    """Rule-based NL explanation from claim + mitigation stats."""
    qid = row["question_id"]
    tc = row["total_claims"]
    sc = row["supported_claims"]
    cc = row["contradicted_claims"]
    nc = row["neutral_claims"]
    hr = row["hallucination_risk"]
    resolved = row.get("resolved_claims", 0)
    corrected = row.get("corrected_claims", 0)
    reretrieved = row.get("reretrieved_claims", 0)
    human_rev = row.get("human_review_claims", 0)
    risk = row["risk_level"]

    parts = [f"The generated answer contains {tc} factual claim(s)."]

    if sc == tc:
        parts.append(f"All {tc} claim(s) are fully supported by the retrieved evidence.")
    else:
        parts.append(f"{sc} of {tc} claim(s) are directly supported; "
                     f"{cc} contradict(s) the evidence; "
                     f"{nc} lack(s) sufficient evidence.")

    if corrected > 0:
        parts.append(f"{corrected} claim(s) were corrected using evidence-grounded rewriting "
                     "and verified with NLI.")
    if reretrieved > 0:
        parts.append(f"{reretrieved} claim(s) were resolved after targeted re-retrieval "
                     "of additional evidence.")
    if human_rev > 0:
        parts.append(f"{human_rev} claim(s) could not be automatically resolved and were "
                     "escalated to human expert review.")

    faith_val = row.get("sent_faithfulness", None)
    if faith_val is not None:
        parts.append(f"Sentence-level faithfulness: {faith_val:.1%}.")

    if risk == "LOW":
        parts.append("Overall risk: LOW — no unresolved claim was detected in the available evidence trace.")
    elif risk == "MEDIUM":
        parts.append("Overall risk: MEDIUM — review recommended before relying on the answer.")
    else:
        parts.append("Overall risk: HIGH — human expert review required before use.")

    return " ".join(parts)

xai_answer_dashboard_df["nl_explanation"] = xai_answer_dashboard_df.apply(generate_nl_explanation, axis=1)

print(f"Generated NL explanations for {len(xai_answer_dashboard_df)} questions.")
print()
# Show a few examples
for _, r in xai_answer_dashboard_df.head(3).iterrows():
    print(f"[Q{r['question_id']}] Risk={r['risk_level']}")
    print(f"  {r['nl_explanation']}")
    print()


Generated NL explanations for 100 questions.

[Q0] Risk=HIGH
  The generated answer contains 2 factual claim(s). 1 of 2 claim(s) are directly supported; 0 contradict(s) the evidence; 1 lack(s) sufficient evidence. 1 claim(s) were corrected using evidence-grounded rewriting and verified with NLI. 1 claim(s) were resolved after targeted re-retrieval of additional evidence. Sentence-level faithfulness: 75.0%. Overall risk: HIGH — human expert review required before use.

[Q1] Risk=LOW
  The generated answer contains 8 factual claim(s). All 8 claim(s) are fully supported by the retrieved evidence. Sentence-level faithfulness: 100.0%. Overall risk: LOW — no unresolved claim was detected in the available evidence trace.

[Q2] Risk=LOW
  The generated answer contains 4 factual claim(s). All 4 claim(s) are fully supported by the retrieved evidence. Sentence-level faithfulness: 100.0%. Overall risk: LOW — no unresolved claim was detected in the available evidence trace.



---
# 5.8 Chart A — Risk Distribution

Pie chart showing the distribution of risk levels (LOW / MEDIUM / HIGH) across the 100 evaluated questions.


In [ ]:
# Section 5.8 - Chart A: Risk Distribution (pie chart)
import matplotlib
matplotlib.use("Agg")

fig, ax = plt.subplots(figsize=(6, 6))
risk_counts = xai_answer_dashboard_df["risk_level"].value_counts().reindex(risk_order, fill_value=0)
colors = [risk_palette[l] for l in risk_order]

wedges, texts, autotexts = ax.pie(
    risk_counts.values, labels=risk_order, autopct="%1.1f%%",
    colors=colors, startangle=90, textprops={"fontsize": 12},
)
for t in autotexts:
    t.set_fontweight("bold")
ax.set_title("Risk Distribution Across 100 Questions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(XAI_DIR / "chart_a_risk_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("Chart A saved.")


Chart A saved.


---
# 5.9 Chart B — Claim Verification Funnel

Stacked bar showing the initial claim verification outcomes (SUPPORTED / CONTRADICTED / NOT_ENOUGH_EVIDENCE)
and the final outcomes after mitigation.


In [ ]:
# Section 5.9 - Chart B: Claim Verification Funnel
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Initial distribution
init_counts = xai_claim_explanations_df["verification_label"].value_counts()
init_labels = ["SUPPORTED", "CONTRADICTED", "NOT_ENOUGH_EVIDENCE"]
init_vals = [init_counts.get(l, 0) for l in init_labels]
init_colors = ["#2ecc71", "#e74c3c", "#f39c12"]

axes[0].barh(init_labels, init_vals, color=init_colors, edgecolor="white")
for i, v in enumerate(init_vals):
    axes[0].text(v + 3, i, f"{v} ({v/len(xai_claim_explanations_df)*100:.1f}%)", va="center", fontsize=11)
axes[0].set_xlim(0, max(init_vals) * 1.25)
axes[0].set_title("Before Mitigation", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Number of claims")

# Final distribution
final_status_map = {
    "RESOLVED_SUPPORTED": "SUPPORTED",
    "RESOLVED_GROUNDED": "CORRECTED",
    "RESOLVED_AFTER_RERETRIEVAL": "RE_RETRIEVED",
    "RESOLVED_CORRECTED": "CORRECTED",
    "RESOLVED_CORRECTED_AFTER_RERETRIEVAL": "CORRECTED",
    "UNRESOLVED_HUMAN_REVIEW": "HUMAN_REVIEW",
}
xai_claim_explanations_df["final_label"] = xai_claim_explanations_df["final_status"].map(final_status_map).fillna("UNKNOWN")
final_counts = xai_claim_explanations_df["final_label"].value_counts()
final_labels = ["SUPPORTED", "CORRECTED", "RE_RETRIEVED", "HUMAN_REVIEW"]
final_vals = [final_counts.get(l, 0) for l in final_labels]
final_colors = ["#2ecc71", "#3498db", "#9b59b6", "#e74c3c"]

axes[1].barh(final_labels, final_vals, color=final_colors, edgecolor="white")
for i, v in enumerate(final_vals):
    axes[1].text(v + 3, i, f"{v} ({v/len(xai_claim_explanations_df)*100:.1f}%)", va="center", fontsize=11)
axes[1].set_xlim(0, max(final_vals) * 1.25)
axes[1].set_title("After Mitigation", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Number of claims")

fig.suptitle("Claim Verification Funnel", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(XAI_DIR / "chart_b_claim_funnel.png", dpi=150, bbox_inches="tight")
plt.close()
print("Chart B saved.")


Chart B saved.


---
# 5.10 Chart C — Claim-Evidence Graph View

For the case-study question, render a bipartite-style graph connecting claims to their
evidence chunks using `matplotlib` (no external graph library).


In [ ]:
# Section 5.10 - Chart C: Claim-Evidence Graph View (matplotlib)
# Faithful lifecycle: Claim -> Evidence -> Verification -> Mitigation Action -> Final Status
case_ev_graph = xai_evidence_attribution_df[xai_evidence_attribution_df["question_id"] == case_qid].copy()
case_cl_graph = xai_claim_explanations_df[xai_claim_explanations_df["question_id"] == case_qid].copy()

fig, ax = plt.subplots(figsize=(20, 8))

claim_ids = case_cl_graph["claim_id"].tolist()
claim_texts = [f"[{cid}] {str(c)[:55]}..." for cid, c in zip(claim_ids, case_cl_graph["claim"])]

y_claims = np.linspace(0.92, 0.08, len(claim_ids))

# --- colour maps ---
label_colors = {"SUPPORTED": "#2ecc71", "CONTRADICTED": "#e74c3c", "NOT_ENOUGH_EVIDENCE": "#f39c12"}
verification_labels = {"SUPPORTED": "Entailed", "CONTRADICTED": "Contradicted", "NOT_ENOUGH_EVIDENCE": "No evidence"}
action_labels = {
    "RESOLVED_SUPPORTED": "Supported\n(no action)",
    "RESOLVED_GROUNDED": "Grounding\nrewrite",
    "RESOLVED_CORRECTED": "Corrected",
    "RESOLVED_AFTER_RERETRIEVAL": "Re-retrieved",
    "RESOLVED_CORRECTED_AFTER_RERETRIEVAL": "Corrected +\nRe-retrieved",
    "UNRESOLVED_HUMAN_REVIEW": "Human\nreview",
}
action_colors = {
    "RESOLVED_SUPPORTED": "#2ecc71",
    "RESOLVED_GROUNDED": "#3498db",
    "RESOLVED_CORRECTED": "#3498db",
    "RESOLVED_AFTER_RERETRIEVAL": "#9b59b6",
    "RESOLVED_CORRECTED_AFTER_RERETRIEVAL": "#9b59b6",
    "UNRESOLVED_HUMAN_REVIEW": "#e74c3c",
}
status_colors = {
    "RESOLVED_SUPPORTED": "#2ecc71",
    "RESOLVED_GROUNDED": "#3498db",
    "RESOLVED_CORRECTED": "#3498db",
    "RESOLVED_AFTER_RERETRIEVAL": "#9b59b6",
    "RESOLVED_CORRECTED_AFTER_RERETRIEVAL": "#9b59b6",
    "UNRESOLVED_HUMAN_REVIEW": "#e74c3c",
}

# Column x-positions
x_claim  = 0.01   # Col 1: original claim
x_ev     = 0.28   # Col 2: evidence chunk
x_verif  = 0.53   # Col 3: verification label
x_action = 0.67   # Col 4: mitigation action
x_final  = 0.83   # Col 5: final status / escalated

# ---- Col 1: Claim nodes ----
for i, (y, label) in enumerate(zip(y_claims, claim_texts)):
    status = case_cl_graph.iloc[i]["verification_label"]
    color = label_colors.get(status, "#95a5a6")
    ax.add_patch(mpatches.FancyBboxPatch((x_claim, y - 0.03), 0.24, 0.06,
                  boxstyle="round,pad=0.005", facecolor=color, alpha=0.85,
                  edgecolor="black", linewidth=0.5))
    ax.text(x_claim + 0.01, y, label, fontsize=7, va="center", fontfamily="monospace")

# ---- Cols 2-5: per-claim loop ----
for i, (_, cr) in enumerate(case_cl_graph.iterrows()):
    y = y_claims[i]
    cid = cr["claim_id"]
    fs = cr["final_status"]
    vlabel = cr["verification_label"]

    # Col 2: Evidence node
    ev_row = case_ev_graph[case_ev_graph["claim_id"] == cid]
    if fs != "UNRESOLVED_HUMAN_REVIEW" and not ev_row.empty:
        evr = ev_row.iloc[0]
        chunk_id = evr.get("final_evidence_chunk_id", "?")
        alignment = evr.get("evidence_alignment_score", 0)
        ev_text = f"Chunk {chunk_id}  align={alignment:.3f}"
        ax.add_patch(mpatches.FancyBboxPatch((x_ev, y - 0.03), 0.22, 0.06,
                      boxstyle="round,pad=0.005", facecolor="#d5e8f0", alpha=0.85,
                      edgecolor="#2980b9", linewidth=0.5))
        ax.text(x_ev + 0.01, y, ev_text, fontsize=7, va="center", fontfamily="monospace")
    else:
        ax.add_patch(mpatches.FancyBboxPatch((x_ev, y - 0.03), 0.22, 0.06,
                      boxstyle="round,pad=0.005", facecolor="#f5f5f5", alpha=0.5,
                      edgecolor="#bdc3c7", linewidth=0.5))
        ax.text(x_ev + 0.11, y, "No evidence", fontsize=7, va="center", ha="center",
                color="#95a5a6", fontstyle="italic")

    # Edge: claim -> evidence
    ax.annotate("", xy=(x_ev, y), xytext=(x_claim + 0.24, y),
                arrowprops=dict(arrowstyle="-|>", color="#555", lw=1.0))

    # Col 3: Verification label node
    vtext = verification_labels.get(vlabel, vlabel)
    vcolor = label_colors.get(vlabel, "#95a5a6")
    ax.add_patch(mpatches.FancyBboxPatch((x_verif, y - 0.03), 0.12, 0.06,
                  boxstyle="round,pad=0.005", facecolor=vcolor, alpha=0.75,
                  edgecolor="black", linewidth=0.5))
    ax.text(x_verif + 0.06, y, vtext, fontsize=7, va="center", ha="center",
            fontweight="bold", color="white")

    # Edge: evidence -> verification
    ax.annotate("", xy=(x_verif, y), xytext=(x_ev + 0.22, y),
                arrowprops=dict(arrowstyle="-|>", color="#555", lw=1.0))

    # Col 4: Mitigation action node
    albl = action_labels.get(fs, fs)
    acol = action_colors.get(fs, "#95a5a6")
    ax.add_patch(mpatches.FancyBboxPatch((x_action, y - 0.03), 0.12, 0.06,
                  boxstyle="round,pad=0.005", facecolor=acol, alpha=0.7,
                  edgecolor="black", linewidth=0.5))
    ax.text(x_action + 0.06, y, albl, fontsize=6.5, va="center", ha="center",
            fontweight="bold", color="white")

    # Edge: verification -> action
    ax.annotate("", xy=(x_action, y), xytext=(x_verif + 0.12, y),
                arrowprops=dict(arrowstyle="-|>", color="#555", lw=1.0))

    # Col 5: Final status node
    if fs == "UNRESOLVED_HUMAN_REVIEW":
        ax.text(x_final + 0.08, y, "ESCALATED TO\nHUMAN EXPERT", fontsize=7.5,
                va="center", ha="center", color="#e74c3c", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#fce4e4", edgecolor="#e74c3c"))
    else:
        final_claim = str(cr.get("final_claim", cr["claim"]))[:60]
        if len(str(cr.get("final_claim", cr["claim"]))) > 60:
            final_claim += "..."
        scol = status_colors.get(fs, "#95a5a6")
        ax.add_patch(mpatches.FancyBboxPatch((x_final, y - 0.03), 0.16, 0.06,
                      boxstyle="round,pad=0.005", facecolor=scol, alpha=0.55,
                      edgecolor="black", linewidth=0.5))
        ax.text(x_final + 0.01, y, final_claim, fontsize=6, va="center", fontfamily="monospace")

    # Edge: action -> final status
    ax.annotate("", xy=(x_final, y), xytext=(x_action + 0.12, y),
                arrowprops=dict(arrowstyle="-|>", color="#555", lw=1.0))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title(f"Claim Lifecycle Graph  (Question {case_qid})",
             fontsize=13, fontweight="bold")

# Column labels
col_labels = [
    (0.13, "Claim"), (0.39, "Evidence"), (0.59, "Verification"),
    (0.73, "Mitigation Action"), (0.91, "Final Status"),
]
for pos, txt in col_labels:
    ax.text(pos, 0.98, txt, fontsize=10, fontweight="bold", ha="center", va="top",
            transform=ax.transAxes)

plt.tight_layout()
plt.savefig(XAI_DIR / "chart_c_claim_evidence_graph.png", dpi=150, bbox_inches="tight")
plt.close()
print("Chart C saved.")


Chart C saved.


---
# 5.11 Chart D — Mitigation Effectiveness

Bar chart comparing `mitigation_success_rate` and `unresolved_rate` across risk levels.


In [ ]:
# Section 5.11 - Chart D: Mitigation Effectiveness by Risk Level (rates vs counts separated)
mitig_stats = xai_answer_dashboard_df.groupby("risk_level", observed=False).agg(
    avg_success=("mitigation_success_rate", "mean"),
    avg_unresolved=("unresolved_rate", "mean"),
    avg_corrected=("corrected_claims", "mean"),
    avg_reretrieved=("reretrieved_claims", "mean"),
).reindex(risk_order)

x = np.arange(len(risk_order))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: rates (all percentage-scale, comparable units)
w = 0.28
bars1 = ax1.bar(x - w, mitig_stats["avg_success"] * 100, w, label="Mitigation success rate (%)",
                color="#2ecc71", edgecolor="white")
bars2 = ax1.bar(x,     mitig_stats["avg_unresolved"] * 100, w, label="Unresolved rate (%)",
                color="#e74c3c", edgecolor="white")
ax1.set_ylabel("Rate (%)", fontsize=12)
ax1.set_title("Mitigation Rates by Risk Level", fontsize=13, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(risk_order, fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(axis="y", alpha=0.3)
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.5, f"{h:.1f}%", ha="center", fontsize=9)

# Right panel: counts (claim counts per question, comparable units)
bars3 = ax2.bar(x - w/2, mitig_stats["avg_corrected"], w, label="Avg corrected claims",
                color="#3498db", edgecolor="white")
bars4 = ax2.bar(x + w/2, mitig_stats["avg_reretrieved"], w, label="Avg re-retrieved claims",
                color="#9b59b6", edgecolor="white")
ax2.set_ylabel("Average claims per question", fontsize=12)
ax2.set_title("Mitigation Counts by Risk Level", fontsize=13, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(risk_order, fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(axis="y", alpha=0.3)
for bars in [bars3, bars4]:
    for bar in bars:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.02, f"{h:.2f}", ha="center", fontsize=9)

fig.suptitle("Mitigation Effectiveness by Risk Level", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(XAI_DIR / "chart_d_mitigation_effectiveness.png", dpi=150, bbox_inches="tight")
plt.close()
print("Chart D saved.")


Chart D saved.


---
# 5.12 Chart E — Evidence Alignment Heatmap

Heatmap of NLI scores and biomedical similarity scores across claims, grouped by risk level.


In [ ]:
# Section 5.12 - Chart E: Evidence Alignment Heatmap
# Merge claims with dashboard risk_level
heatmap_data = xai_claim_explanations_df.merge(
    xai_answer_dashboard_df[["question_id", "risk_level"]], on="question_id", how="left"
)

# Pivot: rows=questions (sorted by risk), columns=claims within each question
# Use a simpler approach: aggregate per question
q_heat = heatmap_data.groupby("question_id").agg(
    mean_nli=("nli_score", "mean"),
    mean_biosim=("bio_similarity", "mean"),
    mean_alignment=("nli_score", lambda x: x.mean()),
    risk_level=("risk_level", "first"),
).reset_index()

# Sort by risk level then by mean alignment
risk_sort = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
q_heat["risk_sort"] = q_heat["risk_level"].map(risk_sort)
q_heat = q_heat.sort_values(["risk_sort", "mean_nli"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 10))
matrix_data = q_heat[["mean_nli", "mean_biosim"]].values.T
col_labels = [f"Q{int(q)}" for q in q_heat["question_id"]]

sns.heatmap(
    matrix_data, ax=ax, cmap="RdYlGn", vmin=0, vmax=1,
    xticklabels=col_labels, yticklabels=["NLI Score", "BioSimilarity"],
    cbar_kws={"label": "Score"},
    linewidths=0.3,
)
ax.set_title("Evidence Alignment Scores per Question (sorted by risk level)", fontsize=13, fontweight="bold")
ax.set_xlabel("Question ID", fontsize=11)
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(XAI_DIR / "chart_e_alignment_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Chart E saved.")


Chart E saved.


---
# 5.13 Charts F & G — Ablation Comparison + Overview Dashboard


In [ ]:
# Section 5.13a - Chart F: Ablation Comparison
fig, ax = plt.subplots(figsize=(10, 5))

# Parse numeric values from ablation_comparison
abl = ablation_df.copy()
abl["without_num"] = abl["without_control"].apply(
    lambda x: float(str(x).replace("%", "").replace("+", "").strip()) if pd.notna(x) else 0
)
abl["with_num"] = abl["with_control"].apply(
    lambda x: float(str(x).replace("%", "").replace("+", "").strip()) if pd.notna(x) else 0
)

# Select key metrics for the chart
key_metrics = ["Supported claim rate", "Contradicted claim rate", "Unsupported claim rate (HR)",
               "Faithfulness", "FActScore", "Answer-level HR"]
abl_key = abl[abl["metric"].isin(key_metrics)].copy()

x = np.arange(len(abl_key))
width = 0.35

bars1 = ax.bar(x - width/2, abl_key["without_num"], width, label="Without Hallucination Control",
               color="#e74c3c", alpha=0.8, edgecolor="white")
bars2 = ax.bar(x + width/2, abl_key["with_num"], width, label="With Hallucination Control",
               color="#2ecc71", alpha=0.8, edgecolor="white")

ax.set_ylabel("Score / Rate", fontsize=12)
ax.set_title("Ablation: Without vs With Hallucination Control", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(abl_key["metric"], rotation=25, ha="right", fontsize=10)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(XAI_DIR / "chart_f_ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Chart F saved.")


Chart F saved.


In [ ]:
# Section 5.13b - Chart G: Comprehensive Overview Dashboard (4-panel)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Risk distribution bar
risk_counts = xai_answer_dashboard_df["risk_level"].value_counts().reindex(risk_order, fill_value=0)
axes[0, 0].bar(risk_order, risk_counts.values, color=[risk_palette[l] for l in risk_order], edgecolor="white")
axes[0, 0].set_title("A. Risk Distribution", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Number of questions")
for i, v in enumerate(risk_counts.values):
    axes[0, 0].text(i, v + 0.5, str(v), ha="center", fontweight="bold", fontsize=11)

# Panel 2: Support rate distribution
axes[0, 1].hist(xai_answer_dashboard_df["support_rate"] * 100, bins=20, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[0, 1].axvline(xai_answer_dashboard_df["support_rate"].mean() * 100, color="red", linestyle="--",
                    label=f"Mean = {xai_answer_dashboard_df['support_rate'].mean()*100:.1f}%")
axes[0, 1].set_title("B. Support Rate Distribution", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("Support rate (%)")
axes[0, 1].set_ylabel("Count")
axes[0, 1].legend()

# Panel 3: Mitigation outcomes
mitig_labels = ["Clean", "Mitigated", "Partially Resolved", "Review Needed"]
status_map_count = xai_answer_dashboard_df["final_answer_status"].value_counts()
mitig_vals = [status_map_count.get(s, 0) for s in ["CLEAN", "MITIGATED", "PARTIALLY_RESOLVED", "REVIEW_NEEDED"]]
mitig_colors = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c"]
axes[1, 0].barh(mitig_labels, mitig_vals, color=mitig_colors, edgecolor="white")
axes[1, 0].set_title("C. Answer Status After Mitigation", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("Number of questions")
for i, v in enumerate(mitig_vals):
    axes[1, 0].text(v + 0.3, i, str(v), va="center", fontweight="bold", fontsize=11)

# Panel 4: Faithfulness vs Support Rate scatter
axes[1, 1].scatter(xai_answer_dashboard_df["support_rate"] * 100, xai_answer_dashboard_df.get("sent_faithfulness", pd.Series([0]*len(xai_answer_dashboard_df))) * 100,
                   c=xai_answer_dashboard_df["risk_level"].map(risk_palette).fillna("#95a5a6"),
                   alpha=0.7, edgecolor="black", linewidth=0.5, s=60)
axes[1, 1].set_title("D. Support Rate vs Faithfulness", fontsize=12, fontweight="bold")
axes[1, 1].set_xlabel("Claim support rate (%)")
axes[1, 1].set_ylabel("Sentence faithfulness (%)")
# legend for colors
for level in risk_order:
    axes[1, 1].scatter([], [], c=risk_palette[level], label=level, s=40, edgecolor="black", linewidth=0.5)
axes[1, 1].legend(title="Risk level", fontsize=9)

fig.suptitle("XAI Overview Dashboard", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(XAI_DIR / "chart_g_overview_dashboard.png", dpi=150, bbox_inches="tight")
plt.close()
print("Chart G saved.")


Chart G saved.


---
# 5.13c Chart H — Final Pipeline Outcome

Pie chart: automatically resolved/acceptable claims vs those requiring human expert review.


In [ ]:
# Section 5.13c - Chart H: Final Pipeline Outcome
total_claims_n = len(xai_claim_explanations_df)
auto_resolved  = (xai_claim_explanations_df["final_status"] != "UNRESOLVED_HUMAN_REVIEW").sum()
human_review   = (xai_claim_explanations_df["final_status"] == "UNRESOLVED_HUMAN_REVIEW").sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: claim-level
sizes_c = [auto_resolved, human_review]
labels_c = [
    f"Resolved automatically\n({auto_resolved} claims, {auto_resolved/total_claims_n*100:.1f}%)",
    f"Human review\n({human_review} claims, {human_review/total_claims_n*100:.1f}%)",
]
wedges1, _, autotexts1 = ax1.pie(
    sizes_c, labels=labels_c, autopct="", colors=["#2ecc71", "#e74c3c"],
    startangle=90, textprops={"fontsize": 11},
)
for t in autotexts1:
    t.set_fontweight("bold")
ax1.set_title("Claim-Level Outcome", fontsize=13, fontweight="bold")

# Right: question-level
n_with_review = int(xai_answer_dashboard_df["needs_human_review"].sum())
n_clean = len(xai_answer_dashboard_df) - n_with_review
sizes_q = [n_clean, n_with_review]
labels_q = [
    f"Acceptable ({n_clean} questions, {n_clean/len(xai_answer_dashboard_df)*100:.0f}%)",
    f"Needs review ({n_with_review} questions, {n_with_review/len(xai_answer_dashboard_df)*100:.0f}%)",
]
wedges2, _, autotexts2 = ax2.pie(
    sizes_q, labels=labels_q, autopct="", colors=["#2ecc71", "#e74c3c"],
    startangle=90, textprops={"fontsize": 11},
)
for t in autotexts2:
    t.set_fontweight("bold")
ax2.set_title("Question-Level Outcome", fontsize=13, fontweight="bold")

fig.suptitle("Final Pipeline Outcome: Auto-Resolved vs Human Review",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(XAI_DIR / "chart_h_final_outcome.png", dpi=150, bbox_inches="tight")
plt.close()
print("Chart H saved.")


Chart H saved.


---
# 5.14 Per-Question XAI Report Function

A reusable function that generates a structured text report for any single question.


In [ ]:
# Section 5.14 - Per-Question XAI Report Function

def xai_report(question_id, dashboard_df=None, claims_df=None, evidence_df=None):
    """Return a formatted multi-level XAI report for a single question."""
    if dashboard_df is None:
        dashboard_df = xai_answer_dashboard_df
    if claims_df is None:
        claims_df = xai_claims
    if evidence_df is None:
        evidence_df = xai_evidence

    row = dashboard_df[dashboard_df["question_id"] == question_id]
    if row.empty:
        return f"Question {question_id} not found."
    row = row.iloc[0]

    lines = []
    sep = "=" * 72
    lines.append(sep)
    lines.append(f"  XAI REPORT — Question {question_id}")
    lines.append(sep)

    # Level 1
    lines.append("")
    lines.append("[LEVEL 1 — QUESTION DASHBOARD]")
    lines.append(f"  Question          : {row['question'][:200]}")
    lines.append(f"  Gold decision     : {row.get('gold_final_decision', 'N/A')}")
    lines.append(f"  Mitigated answer  : {str(row.get('mitigated_answer', ''))[:300]}")
    lines.append(f"  Total claims      : {row['total_claims']}")
    lines.append(f"  Supported         : {row['supported_claims']}")
    lines.append(f"  Contradicted      : {row['contradicted_claims']}")
    lines.append(f"  Neutral           : {row['neutral_claims']}")
    lines.append(f"  Hall. risk        : {row['hallucination_risk']:.3f}")
    lines.append(f"  Risk level        : {row['risk_level']}")
    lines.append(f"  Answer status     : {row['final_answer_status']}")
    lines.append(f"  Recommended       : {row['recommended_action']}")
    lines.append(f"  Faithfulness      : {row.get('sent_faithfulness', 'N/A')}")
    lines.append(f"  Explanation       : {row['explanation_summary']}")
    lines.append(f"  Plain language    : {row['plain_language']}")

    # Level 2
    q_claims = claims_df[claims_df["question_id"] == question_id]
    lines.append("")
    lines.append(f"[LEVEL 2 — CLAIM EXPLANATIONS] ({len(q_claims)} claims)")
    for _, cr in q_claims.iterrows():
        lines.append(f"  [{cr['claim_id']}] {cr['claim'][:120]}")
        lines.append(f"    Label={cr['verification_label']}  NLI={cr['nli_score']:.3f}  "
                     f"BioSim={cr['bio_similarity']:.3f}")
        lines.append(f"    Final={cr['final_status']}  Evidence={cr['evidence_source_type']}  "
                     f"Attempts={cr['num_attempts']}")

    # Level 3
    q_ev = evidence_df[evidence_df["question_id"] == question_id]
    lines.append("")
    lines.append(f"[LEVEL 3 — EVIDENCE ATTRIBUTION] ({len(q_ev)} records)")
    for _, er in q_ev.iterrows():
        lines.append(f"  [{er['claim_id']}] chunk={er['final_evidence_chunk_id']}  "
                     f"alignment={er['evidence_alignment_score']:.3f}")
        excerpt = str(er.get("evidence_excerpt", ""))[:120]
        if excerpt and excerpt != "nan":
            lines.append(f"    {excerpt}")

    lines.append("")
    lines.append(sep)
    return "\n".join(lines)


# Demo: print report for the case study question
print(xai_report(case_qid))


  XAI REPORT — Question 0

[LEVEL 1 — QUESTION DASHBOARD]
  Question          : Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?
  Gold decision     : yes
  Mitigated answer  : 36.4% of Pap smears with glandular cell abnormalities were flagged as "review for abnormality" on rapid prescreening. Thirteen of 24 Pap smears (54.2%) from patients who had histologic follow-up and were found to harbor a high-grade squamous intraepithelial lesion or carcinoma had been flagged as re
  Total claims      : 2
  Supported         : 1
  Contradicted      : 0
  Neutral           : 1
  Hall. risk        : 0.500
  Risk level        : HIGH
  Answer status     : MITIGATED
  Recommended       : FLAG_AND_REVIEW
  Faithfulness      : 0.75
  Explanation       : 2 claims identified; 1 supported, 0 contradicted, 1 unresolved. 1 claim(s) corrected via evidence-grounded rewrite. 1 claim(s) resolved through re-retrieval.
  Plain language    : 1 of 2 claims are supported. Multi

---
# 5.15 Save All XAI Outputs

Persist the three XAI dataframes, the full text report, and all charts to `data/outputs/xai/`.


In [ ]:
# Section 5.15 - Save All XAI Outputs
# ---- CSVs ----
xai_answer_dashboard_df.to_csv(XAI_DASHBOARD_CSV, index=False)
xai_claim_explanations_df.to_csv(XAI_CLAIMS_CSV, index=False)
xai_evidence_attribution_df.to_csv(XAI_EVIDENCE_CSV, index=False)

# ---- Full text report ----
report_lines = []
for qid in sorted(xai_answer_dashboard_df["question_id"].unique()):
    report_lines.append(xai_report(qid))
    report_lines.append("\n\n")

XAI_REPORT_TXT.write_text("\n".join(report_lines), encoding="utf-8")

# ---- Summary ----
print("XAI outputs saved:")
print(f"  Dashboard  : {XAI_DASHBOARD_CSV}  ({xai_answer_dashboard_df.shape[0]} rows, {xai_answer_dashboard_df.shape[1]} cols)")
print(f"  Claims     : {XAI_CLAIMS_CSV}  ({xai_claim_explanations_df.shape[0]} rows, {xai_claim_explanations_df.shape[1]} cols)")
print(f"  Evidence   : {XAI_EVIDENCE_CSV}  ({xai_evidence_attribution_df.shape[0]} rows, {xai_evidence_attribution_df.shape[1]} cols)")
print(f"  Report     : {XAI_REPORT_TXT}")
print()
print("Charts saved:")
for png in sorted(XAI_DIR.glob("chart_*.png")):
    print(f"  {png.name}")
print()
print("XAI module complete.")


XAI outputs saved:
  Dashboard  : d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\xai\xai_answer_dashboard.csv  (100 rows, 36 cols)
  Claims     : d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\xai\xai_claim_explanations.csv  (644 rows, 18 cols)
  Evidence   : d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\xai\xai_evidence_attribution.csv  (644 rows, 15 cols)
  Report     : d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\xai\xai_report.txt

Charts saved:
  chart_a_risk_distribution.png
  chart_b_claim_funnel.png
  chart_c_claim_evidence_graph.png
  chart_d_mitigation_effectiveness.png
  chart_e_alignment_heatmap.png
  chart_f_ablation_comparison.png
  chart_g_overview_dashboard.png
  chart_h_final_outcome.png

XAI module complete.
